# AutoE2E in the AlpaSim closed loop: a walkthrough

This notebook follows AutoE2E through one closed-loop rollout in NVIDIA AlpaSim, using the driver plugin in
`Model/plugins/alpasim_driver/`. It ends with the reward signal that closed-loop fine-tuning is meant to train on.
Read it next to the [overview](overview.md).

| Part | What it shows | Status |
| :--- | :--- | :--- |
| 1–3 | Setup, the policy checkpoint, and the camera rig the driver loads | implemented |
| 4 | One driving decision, offline: AlpaSim input, model inputs, controls, waypoints | implemented |
| 5 | Launching a closed-loop rollout (off by default) | implemented |
| 6 | What happened in a rollout: AlpaSim's own video, bird's-eye views on the scene map, ego vs. recording, infractions, camera frames | implemented |
| 7 | The rollout's phase-1 reward | implemented |
| 8 | The per-decision table an RL update would consume | the RL update itself is **planned**, not implemented |

**Prerequisites** (see the overview's "Run it yourself"):

- an AlpaSim checkout with the plugin installed; use its `.venv` as the kernel
- a released AutoE2E checkpoint; this notebook uses nuPlan Epoch 5
- for parts 6–8, a finished rollout of `driver=autoe2e_nuplan`

Paths come from the environment variables `ALPASIM_ROOT`, `AUTOE2E_CHECKPOINT` and `ROLLOUT_LOG_DIR`. Their defaults
follow the layout in the plugin README.

## 1. Setup

In [ ]:
import collections
import contextlib
import io
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml


def find_repo(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "Model" / "plugins" / "alpasim_driver").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the auto_e2e repository, or set AUTOE2E_REPO.")


REPO = Path(os.environ["AUTOE2E_REPO"]) if "AUTOE2E_REPO" in os.environ else find_repo(Path.cwd().resolve())
PLUGIN_DIR = REPO / "Model" / "plugins" / "alpasim_driver"
CONFIG_DIR = PLUGIN_DIR / "alpasim_autoe2e" / "configs" / "driver"
ALPASIM_ROOT = Path(os.environ.get("ALPASIM_ROOT", REPO.parent / "alpasim"))
CHECKPOINT = Path(
    os.environ.get("AUTOE2E_CHECKPOINT", ALPASIM_ROOT / "data" / "drivers" / "autoe2e" / "checkpoint.pt")
)
ROLLOUT_LOG_DIR = Path(os.environ.get("ROLLOUT_LOG_DIR", ALPASIM_ROOT / "outputs" / "autoe2e_closed_loop_run"))
RUN_SIM = os.environ.get("RUN_SIM", "0") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Import the plugin, the AutoE2E model code and the plugin's tools the way the tests do.
for path in (REPO / "Model", PLUGIN_DIR, PLUGIN_DIR / "tools"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"repo          {REPO}")
print(f"AlpaSim       {ALPASIM_ROOT}")
print(f"checkpoint    {CHECKPOINT}  (found: {CHECKPOINT.is_file()})")
print(f"rollout logs  {ROLLOUT_LOG_DIR}  (found: {ROLLOUT_LOG_DIR.is_dir()})")
print(f"device        {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else ""))

## 2. The policy

The driver rebuilds AutoE2E from the `config` saved in the training checkpoint, the same reconstruction as
`Platform/pipelines/inference.py`, so the architecture always matches the weights. The released checkpoints are the
reactive BEVFormer T8 model:

- a ResNet-50 camera backbone
- BEVFormer v2 view fusion over the current frame plus seven history frames
- a map and route fusion stage
- a GRU planner that predicts 64 steps of (acceleration, curvature) at 10 Hz, 6.4 s in total

In [ ]:
from alpasim_autoe2e.plugin import load_checkpoint_model

model, config = load_checkpoint_model(CHECKPOINT, torch.device(DEVICE))
view_fusion = config["view_fusion_kwargs"]
policy_summary = {
    "view fusion": view_fusion["architecture"],
    "backbone": config["backbone"],
    "camera slots": ", ".join(config["camera_slots"]),
    "tile size": f"{view_fusion['image_size']} px, front also {view_fusion['front_image_size']} px",
    "BEV grid": f"{view_fusion['bev_h']} x {view_fusion['bev_w']}",
    "planner": config["planner_mode"],
    "route conditioning": config["enable_route_conditioning"],
    "map / route channels": f"{config['map_context_channels']} / {config['route_channels']}",
    "parameters": f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M",
}
del model  # the driver in part 4 loads its own copy
torch.cuda.empty_cache()
pd.Series(policy_summary, name=CHECKPOINT.parent.name).to_frame()

## 3. The camera rig

`driver=autoe2e_nuplan` gives the model the six nuPlan cameras. These are rectified pinhole cameras of vehicle veh-35.

AlpaSim's `extra_cameras` can only override cameras that a NuRec scene already has. So the config renders the
nuPlan cameras on six NVIDIA camera IDs, and `configs/calibration/nuplan_nurec.json` maps them back. That is why the
rear camera is called `camera_front_tele_30fov`.

The driver picks the rig profile whose cameras equal `driver.inference.use_cameras`, then orders the cameras by the
checkpoint's `camera_slots`.

In [ ]:
from alpasim_autoe2e.config import find_rig_profile

driver_cfg = yaml.safe_load((CONFIG_DIR / "autoe2e_nuplan.yaml").read_text())
runtime_cfg = yaml.safe_load((CONFIG_DIR / "autoe2e_nuplan_configs.yaml").read_text())
CAMERAS = driver_cfg["inference"]["use_cameras"]
profile = find_rig_profile(CAMERAS)
rig = profile.ordered_by_slots(config["camera_slots"])


def optical_axis_yaw(camera) -> float:
    forward = camera.camera_to_reference[:3, 2]  # camera z axis in the rig frame (x forward, y left)
    return float(np.arctan2(forward[1], forward[0]))


def horizontal_fov(camera) -> float:
    return float(2 * np.arctan(camera.image_size_wh[0] / (2 * camera.intrinsic[0, 0])))


print(f"rig profile {profile.name}: dataset {profile.dataset_name}, native front tile {profile.native_front_camera_tile}")
pd.DataFrame(
    [
        {
            "slot": camera.slot,
            "AlpaSim camera": camera.logical_id,
            "native size (w x h)": "{} x {}".format(*camera.image_size_wh),
            "x, y, z (m)": tuple(np.round(camera.camera_to_reference[:3, 3], 2)),
            "yaw (deg)": round(np.degrees(optical_axis_yaw(camera)), 1),
            "horizontal FOV (deg)": round(np.degrees(horizontal_fov(camera)), 1),
        }
        for camera in rig
    ]
)

In [ ]:
from matplotlib.patches import Wedge

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for index, camera in enumerate(rig):
    x, y = camera.camera_to_reference[:2, 3]
    yaw, half_fov = np.degrees(optical_axis_yaw(camera)), np.degrees(horizontal_fov(camera)) / 2
    # Plot axes are (right, forward); a rig yaw of 0 (forward) points up, so add 90 degrees.
    ax.add_patch(Wedge((-y, x), 3.0, 90 + yaw - half_fov, 90 + yaw + half_fov, color=f"C{index}", alpha=0.3))
    label_at = (-y - 3.5 * np.sin(np.radians(yaw)), x + 3.5 * np.cos(np.radians(yaw)))
    ax.annotate(camera.slot, label_at, ha="center", va="center", fontsize=8, color=f"C{index}")
ax.add_patch(plt.Rectangle((-0.95, -1.0), 1.9, 4.6, fill=False, color="0.4"))  # rough vehicle outline
ax.set(xlabel="right (m)", ylabel="forward (m)", title="Camera frusta in the rig frame (top view)", aspect="equal")
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 7)
plt.show()

## 4. One decision, offline

AlpaSim calls `AutoE2EDriver.predict(PredictionInput)` every 0.1 s of simulated time. The input carries:

- for each camera, the frames the driver asked for: `context_length: 8` with `subsample_factor: 5`, so 8 frames
  500 ms apart
- the ego pose history in AlpaSim's local frame
- speed and acceleration
- a discrete command and the latest route, both of which AutoE2E ignores today

Here we build a **synthetic** input: flat grey images, and an ego that has been driving a gentle left curve at
8 m/s. It is only for tracing the data path, so the model's answer on grey images means nothing. Parts 6–7 use a
real rollout.

In [ ]:
from alpasim_autoe2e.parser import CONTEXT_FRAMES, egomotion_from_poses, pose_trajectory
from alpasim_autoe2e.plugin import AutoE2EDriver
from alpasim_driver.models.base import CameraFrame, DriveCommand, PredictionInput
from alpasim_grpc.v0 import common_pb2

T0_US = 10_000_000
SPEED_MPS, YAW_RATE = 8.0, 0.05


def pose_at(timestamp_us: int, x: float, y: float, yaw: float) -> common_pb2.PoseAtTime:
    pose = common_pb2.PoseAtTime(timestamp_us=int(timestamp_us))
    pose.pose.vec.x, pose.pose.vec.y = x, y
    pose.pose.quat.z, pose.pose.quat.w = np.sin(yaw / 2), np.cos(yaw / 2)
    return pose


# 10 s of 10 Hz pose history on a circular arc, as the forced-ground-truth warm-up would provide.
radius = SPEED_MPS / YAW_RATE
ego_history = [
    pose_at(t, radius * np.sin(YAW_RATE * t / 1e6), radius * (1 - np.cos(YAW_RATE * t / 1e6)), YAW_RATE * t / 1e6)
    for t in range(0, T0_US + 1, 100_000)
]
frame_times = [T0_US - (CONTEXT_FRAMES - 1 - k) * 500_000 for k in range(CONTEXT_FRAMES)]
sizes = {camera["logical_id"]: (camera["height"], camera["width"]) for camera in runtime_cfg["runtime"]["simulation_config"]["cameras"]}
prediction_input = PredictionInput(
    camera_images={
        camera: [CameraFrame(t, np.full((*sizes[camera], 3), 128, np.uint8)) for t in frame_times] for camera in CAMERAS
    },
    command=DriveCommand.STRAIGHT,
    speed=SPEED_MPS,
    acceleration=0.0,
    ego_pose_history=ego_history,
    inference_seed=0,
    previous_plan=None,
    route=None,
)

driver = AutoE2EDriver(str(CHECKPOINT), CAMERAS, device=DEVICE)
parsed = driver.parser.parse(prediction_input)
print(f"t0 = {parsed.t0_us} us, initial speed {parsed.initial_speed_mps:.2f} m/s")
pd.DataFrame(
    [
        {"model input": name, "shape": tuple(value.shape), "dtype": str(value.dtype).removeprefix("torch.")}
        for name, value in parsed.model_inputs.items()
    ]
)

The parser rebuilds the packed training tensors:

- **Tiles:** the camera frames, squashed to square tiles. The front camera also enters at 1024 px.
- **Projections:** each history projection maps current-ego points into a past image.
- **Egomotion:** 64 rows of speed, acceleration, yaw rate and curvature, derived from the poses.
- **Map and route:** these get zeros with `map_valid = route_valid = False`, the model's camera-only track. AlpaSim
  passes no map, and it passes the route as waypoints, which the plugin does not rasterize yet.

Below is the egomotion the model sees before dataset normalisation.

In [ ]:
egomotion = egomotion_from_poses(pose_trajectory(ego_history), parsed.t0_us, profile.egomotion)
fig, axes = plt.subplots(1, 4, figsize=(15, 2.6), sharex=True)
for ax, values, label in zip(axes, egomotion.T, ["speed (m/s)", "acceleration (m/s²)", "yaw rate (rad/s)", "curvature (1/m)"]):
    ax.plot(np.arange(-64, 0) * 0.1, values)
    ax.set(title=label, xlabel="time before t0 (s)")
fig.tight_layout()
plt.show()

Next, the model runs, and the training rollout (`integrate_controls_torch`) turns its 64 (acceleration, curvature)
controls into waypoints. `predict()` does exactly this. We call its internal model step only to show the raw
controls, and check the result against the public call.

In [ ]:
from training.losses.control_rollout import integrate_controls_torch

start = time.perf_counter()
controls = driver._run_model(parsed).reshape(-1, 2).float().cpu()  # internal step of predict()
if DEVICE == "cuda":
    torch.cuda.synchronize()
print(f"one AutoE2E decision: {time.perf_counter() - start:.1f} s on {DEVICE}")

positions, headings, _ = integrate_controls_torch(controls[None], torch.tensor([parsed.initial_speed_mps]))
prediction = driver.predict(prediction_input)
# BEVFormer's CUDA kernels are not bit-deterministic, so the two calls agree to within a centimetre.
assert np.allclose(prediction.selected_positions[:, :2], positions[0].numpy(), atol=1e-2)
print(f"predict() returns {len(prediction.waypoint_timestamps_us)} waypoints, 0.1 s apart, anchored at t0")

steps = np.arange(1, 65) * 0.1
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].plot(steps, controls[:, 0])
axes[0].set(title="acceleration (m/s²)", xlabel="time after t0 (s)")
axes[1].plot(steps, controls[:, 1])
axes[1].set(title="curvature (1/m)", xlabel="time after t0 (s)")
axes[2].plot(-positions[0, :, 1], positions[0, :, 0], ".-", ms=3)
axes[2].set(title="plan in the rig frame (synthetic input)", xlabel="right (m), not to scale", ylabel="forward (m)")
axes[2].set_xlim(-5, 5)
fig.tight_layout()
plt.show()

## 5. Run a closed-loop rollout (optional)

The wizard starts the AlpaSim services (runtime, renderer, driver, physics, controller) as Docker containers. It then
plays the default NuRec scene for 20 s:

1. **Warm-up (first 7 s):** the ego follows the recording, the driver is skipped, and frames and poses accumulate.
2. **Driving:** AutoE2E decides every 0.1 s.

A rollout takes about 20 minutes of wall time on one A100, so it runs in a detached `tmux` session that outlives
the notebook. Set `RUN_SIM=1` to start it from here. The defaults expect the image `alpasim-base:latest`, built as in
the plugin README; override it with `ALPASIM_BASE_IMAGE`.

In [ ]:
wizard_command = [
    "uv", "run", "--project", "src/wizard", "alpasim_wizard",
    "deploy=local", "topology=1gpu", "driver=autoe2e_nuplan",
    "driver.model.checkpoint_path=/mnt/drivers/autoe2e/checkpoint.pt",
    "defines.base_image=" + os.environ.get("ALPASIM_BASE_IMAGE", "alpasim-base:latest"),
    f"wizard.log_dir={ROLLOUT_LOG_DIR}",
    f"hydra.searchpath=[file://{ALPASIM_ROOT / 'plugins' / 'alpasim_driver' / 'alpasim_autoe2e' / 'configs'}]",
]
print(" ".join(wizard_command[:5]) + " \\\n    " + " \\\n    ".join(wizard_command[5:]))

if RUN_SIM:
    session, log = "autoe2e-closed-loop", ROLLOUT_LOG_DIR.with_suffix(".log")
    shell_command = " ".join(f"'{part}'" for part in wizard_command) + f" 2>&1 | tee -a '{log}'"
    tmux = ["tmux", "new-session", "-d", "-s", session, "-c", str(ALPASIM_ROOT)]
    tmux += ["-e", f"HF_TOKEN={os.environ['HF_TOKEN']}"] if "HF_TOKEN" in os.environ else []
    subprocess.run([*tmux, shell_command], check=True)
    print(f"started tmux session {session}, log {log}")
    while subprocess.run(["tmux", "has-session", "-t", f"={session}"], capture_output=True).returncode == 0:
        time.sleep(30)

completed = sorted(ROLLOUT_LOG_DIR.glob("rollouts/*/*/_complete"))
print(f"{len(completed)} finished rollout(s) under {ROLLOUT_LOG_DIR}")

## 6. What happened in the rollout

AlpaSim writes three things for every rollout, and this part uses all of them:

- `rollout.asl`: every message exchanged in the loop, including the camera frames, the poses of every agent, the
  routes and AutoE2E's plans
- `metrics.parquet`: AlpaSim's metrics for every simulation step, including the infractions
- a video: AlpaSim's own visualisation of the rollout

The vector map comes from the scene's USDZ file, as in AlpaSim's own evaluation.

### 6.1 Read the rollout

In [ ]:
import polars as pl
from alpasim_utils.logs import async_read_pb_log
from score_rollout import se3, to_trajectory


def yaw_of(quat) -> float:
    return float(np.arctan2(2 * (quat.w * quat.z + quat.x * quat.y), 1 - 2 * (quat.y**2 + quat.z**2)))


rollout_logs = sorted(ROLLOUT_LOG_DIR.glob("rollouts/*/*/rollout.asl"))
if not rollout_logs:
    raise FileNotFoundError(f"No rollout.asl under {ROLLOUT_LOG_DIR}: run part 5 (RUN_SIM=1) or the wizard first.")
rollout_asl = rollout_logs[0]
ROLLOUT_DIR = rollout_asl.parent

message_types = collections.Counter()
frames = {}  # (camera, frame end in us) -> the encoded image the driver received
agent_states = []  # (time in us, {actor id: (x, y, yaw) of its box centre in the local frame})
routes = []  # route messages: waypoints in the rig frame at the message's timestamp
plans = []  # AutoE2E's plans in the local frame; the first pose is the ego at decision time
async for message in async_read_pb_log(str(rollout_asl)):
    kind = message.WhichOneof("log_entry")
    message_types[kind] += 1
    entry = getattr(message, kind)
    if kind == "driver_camera_image":
        frames[(entry.camera_image.logical_id, int(entry.camera_image.frame_end_us))] = entry.camera_image.image_bytes
    elif kind == "actor_poses":
        states = {a.actor_id: (a.actor_pose.vec.x, a.actor_pose.vec.y, yaw_of(a.actor_pose.quat)) for a in entry.actor_poses}
        agent_states.append((int(entry.timestamp_us), states))
    elif kind == "route_request":
        routes.append(entry.route)
    elif kind == "driver_return":
        plans.append(list(entry.trajectory.poses))
    elif kind == "rollout_metadata":
        metadata = entry

recorded = to_trajectory(metadata.ego_rig_recorded_ground_truth_trajectory.poses)
box_sizes = {a.actor_id: (a.aabb.size_x, a.aabb.size_y) for a in metadata.actor_definitions.actor_aabb}
RIG_TO_BOX_M = metadata.transform_ego_coords_rig_to_aabb.vec.x  # the ego's box centre is this far ahead of its rig

print(rollout_asl.relative_to(ROLLOUT_LOG_DIR))
print(f"{len(plans)} AutoE2E decisions, {len(agent_states)} simulation steps, {len(box_sizes) - 1} other agents, "
      f"{len(frames)} camera frames")
pd.Series(message_types, name="messages").sort_values(ascending=False).to_frame()

Next, the timelines. The poses of all agents, the ego included, are logged at the centre of their boxes. The
recording and the plans use the ego's rig instead, which sits `RIG_TO_BOX_M` behind the box centre.

Times below are seconds after the first AutoE2E decision, the **takeover** at the end of the 7 s warm-up. The plots
mark four moments with the same colours:

- the takeover
- the first step with the ego more than 3 m from the recording, where the phase-1 reward truncates
- the first off-road step
- the end

In [ ]:
T0 = int(plans[0][0].timestamp_us)


def seconds(t_us):
    return (np.asarray(t_us, dtype=np.float64) - T0) / 1e6


def rig_pose(matrix) -> tuple[float, float, float]:
    return float(matrix[0, 3]), float(matrix[1, 3]), float(np.arctan2(matrix[1, 0], matrix[0, 0]))


def first_time(times_s, mask):
    hits = np.flatnonzero(np.asarray(mask))
    return float(np.asarray(times_s)[hits[0]]) if hits.size else None


step_us = np.array([t for t, _ in agent_states])
t_steps = seconds(step_us)
ego_box = np.array([states["EGO"] for _, states in agent_states])
ego_rig = ego_box.copy()
ego_rig[:, 0] -= RIG_TO_BOX_M * np.cos(ego_box[:, 2])
ego_rig[:, 1] -= RIG_TO_BOX_M * np.sin(ego_box[:, 2])
recorded_range = recorded.time_range_us
recorded_rig = np.array(
    [rig_pose(se3(recorded, int(np.clip(t, recorded_range.start, recorded_range.stop - 1)))) for t in step_us]
)
deviation = np.hypot(*(ego_rig[:, :2] - recorded_rig[:, :2]).T)
decision_us = np.array([int(plan[0].timestamp_us) for plan in plans])
decision_s = seconds(decision_us)

metrics = (
    pl.read_parquet(ROLLOUT_DIR / "metrics.parquet")
    .with_columns(pl.col("timestamps_us").cast(pl.Int64))
    .pivot(on="name", index="timestamps_us", values="values", aggregate_function="first")
    .sort("timestamps_us")
    .to_pandas()
)
metrics["t"] = seconds(metrics["timestamps_us"])

MOMENT_COLORS = {"takeover": "tab:green", "ego > 3 m from the recording": "tab:orange",
                 "first off-road": "tab:red", "end": "0.3"}
moments = {
    "takeover": 0.0,
    "ego > 3 m from the recording": first_time(t_steps, (t_steps >= 0) & (deviation > 3.0)),
    "first off-road": first_time(metrics["t"], metrics["offroad"] > 0),
    "end": float(t_steps[-1]),
}
moments = {name: t for name, t in moments.items() if t is not None}


def mark_moments(ax, label=False):
    for name, t in moments.items():
        ax.axvline(t, color=MOMENT_COLORS[name], ls="--", lw=1, label=name if label else None)


pd.Series(moments, name="seconds after takeover").round(1).to_frame()

### 6.2 AlpaSim's own view of the rollout

AlpaSim renders every rollout into a video (layout `DEFAULT`):

- **Top left:** a bird's-eye view 20 m around the ego, rotated with it. It shows the lanes (blue), road edges
  (black), the other agents (grey, numbered), the ego (green box), the recorded path (green line), AutoE2E's latest
  plan (orange) and the route.
- **Top right:** AlpaSim's metrics, aggregated so far and at this step.
- **Bottom:** the front camera, with the plan drawn on it.

Below are the frames at the key moments, then the whole rollout as an animation. The video is H.264, which VS Code
cannot play inline, so open the file in a media player to watch it at full resolution.

In [ ]:
import cv2
from IPython.display import Image as DisplayImage
from IPython.display import display
from PIL import Image as PILImage

video_path = sorted(ROLLOUT_DIR.glob("*.mp4"))[0]
capture = cv2.VideoCapture(str(video_path))
video_frames = []
while True:
    ok, frame = capture.read()
    if not ok:
        break
    video_frames.append(frame[..., ::-1])  # BGR to RGB
capture.release()
print(f"{video_path}\n{len(video_frames)} frames, one per simulation step")


def video_frame_at(t_s):
    return video_frames[int(np.argmin(np.abs(t_steps[: len(video_frames)] - t_s)))]


fig, axes = plt.subplots(1, len(moments), figsize=(5 * len(moments), 5.8))
for ax, (name, t_s) in zip(axes, moments.items()):
    ax.imshow(video_frame_at(t_s))
    ax.set_title(f"{name} ({t_s:+.1f} s)", color=MOMENT_COLORS[name])
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# The animation starts 2 s before the takeover and shows every second step, so it plays at simulated real time.
first = int(np.argmin(np.abs(t_steps + 2.0)))
gif_frames = [PILImage.fromarray(frame).resize((360, 400)) for frame in video_frames[first::2]]
gif = io.BytesIO()
gif_frames[0].save(gif, format="GIF", save_all=True, append_images=gif_frames[1:], duration=200, loop=0)
print(f"{len(gif_frames)} frames, {len(gif.getvalue()) / 1e6:.1f} MB")
display(DisplayImage(data=gif.getvalue(), format="gif"))

### 6.3 Bird's-eye view of the whole scene

AlpaSim's map view only covers 20 m around the ego. This plot shows the whole drive on the scene's vector map:

- **Grey:** the recorded drive.
- **Ego:** its rig path, coloured by time.
- **Red crosses:** the steps AlpaSim flagged as off-road.
- **Orange:** a plan every second.
- **Thin lines:** the other agents' paths.

In [ ]:
from alpasim_utils.artifact import Artifact
from eval.video_data import ShapelyMap

scene_id = ROLLOUT_DIR.parent.name
scene_files = sorted((ALPASIM_ROOT / "data" / "nre-artifacts" / "all-usdzs").glob("*.usdz"))
scene = next(Artifact(source=str(path)) for path in scene_files if Artifact(source=str(path)).scene_id == scene_id)
with contextlib.redirect_stderr(io.StringIO()):  # the map loader prints progress bars
    map_lines = ShapelyMap.from_vec_map(scene.map).renderable_linestrings
print(f"{scene_id}: {len(map_lines)} map lines from {Path(scene.source).name}")


def draw_map(ax, xlim, ylim):
    for line in map_lines:
        x0, y0, x1, y1 = line.linestring.bounds
        if x1 < xlim[0] or x0 > xlim[1] or y1 < ylim[0] or y0 > ylim[1]:
            continue
        xy = np.asarray(line.linestring.coords)
        ax.plot(xy[:, 0], xy[:, 1], line.style, lw=line.linewidth, alpha=line.alpha, zorder=1)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")


agent_paths = collections.defaultdict(list)
for _, states in agent_states:
    for actor, (x, y, _) in states.items():
        if actor != "EGO":
            agent_paths[actor].append((x, y))
offroad_us = metrics.loc[metrics["offroad"] > 0, "timestamps_us"].to_numpy()
offroad_xy = ego_rig[np.isin(step_us, offroad_us), :2]

path_xy = np.concatenate([recorded_rig[:, :2], ego_rig[:, :2]])
fig, ax = plt.subplots(figsize=(15, 8))
draw_map(ax, (path_xy[:, 0].min() - 25, path_xy[:, 0].max() + 25), (path_xy[:, 1].min() - 25, path_xy[:, 1].max() + 25))
for xy in agent_paths.values():
    xy = np.array(xy)
    ax.plot(xy[:, 0], xy[:, 1], color="0.55", lw=0.8, alpha=0.6, zorder=2)
ax.plot(recorded_rig[:, 0], recorded_rig[:, 1], color="0.45", lw=6, alpha=0.5, zorder=3, label="recorded drive")
for index in range(0, len(plans), 10):
    plan_xy = np.array([[p.pose.vec.x, p.pose.vec.y] for p in plans[index]])
    ax.plot(plan_xy[:, 0], plan_xy[:, 1], color="tab:orange", lw=1, zorder=4)
points = ax.scatter(ego_rig[:, 0], ego_rig[:, 1], c=t_steps, cmap="viridis", s=10, zorder=5)
ax.scatter(offroad_xy[:, 0], offroad_xy[:, 1], color="tab:red", marker="x", s=60, zorder=6, label="off-road (AlpaSim)")
for name, t_s in moments.items():
    step = int(np.argmin(np.abs(t_steps - t_s)))
    ax.scatter(*ego_rig[step, :2], s=160, facecolor="none", edgecolor=MOMENT_COLORS[name], lw=2.5, zorder=7, label=name)
ax.plot([], [], color="tab:orange", lw=1, label="plan, every 1 s")
ax.plot([], [], color="0.55", lw=0.8, label="other agents")
fig.colorbar(points, ax=ax, shrink=0.7, label="ego: seconds after takeover")
ax.set(title="The whole drive on the scene map", xlabel="x (m)", ylabel="y (m)")
ax.legend(loc="lower left", fontsize=8)
plt.show()

### 6.4 Bird's-eye view at the key moments

Each panel shows 60 m around the ego at one moment:

- **Agents:** every agent's box (grey), and the ego (green).
- **Recording:** where the recording had the ego at that time (dashed green box), and the recorded path (green
  line).
- **Plan:** AutoE2E's newest plan (orange).
- **Route:** the route AlpaSim sent the driver (purple dots), which AutoE2E does not use yet.

In [ ]:
from matplotlib.patches import Polygon as PolygonPatch


def box_corners(x, y, yaw, length, width):
    corners = np.array([[length / 2, width / 2], [length / 2, -width / 2], [-length / 2, -width / 2], [-length / 2, width / 2]])
    return corners @ np.array([[np.cos(yaw), np.sin(yaw)], [-np.sin(yaw), np.cos(yaw)]]) + [x, y]


route_us = np.array([int(route.timestamp_us) for route in routes])
ego_yaw_unwrapped = np.unwrap(ego_rig[:, 2])


def route_in_local_frame(t_us):
    route = routes[min(int(np.searchsorted(route_us, t_us)), len(routes) - 1)]
    x = np.interp(route.timestamp_us, step_us, ego_rig[:, 0])
    y = np.interp(route.timestamp_us, step_us, ego_rig[:, 1])
    yaw = np.interp(route.timestamp_us, step_us, ego_yaw_unwrapped)
    waypoints = np.array([[w.x, w.y] for w in route.waypoints])
    return waypoints @ np.array([[np.cos(yaw), np.sin(yaw)], [-np.sin(yaw), np.cos(yaw)]]) + [x, y]


def draw_moment(ax, name, t_s, radius=30.0):
    step = int(np.argmin(np.abs(t_steps - t_s)))
    ex, ey, _ = ego_box[step]
    draw_map(ax, (ex - radius, ex + radius), (ey - radius, ey + radius))
    for actor, (x, y, yaw) in agent_states[step][1].items():
        length, width = box_sizes.get(actor, (4.5, 2.0))
        is_ego = actor == "EGO"
        ax.add_patch(PolygonPatch(box_corners(x, y, yaw, length, width), facecolor="tab:green" if is_ego else "0.75",
                                  edgecolor="k", lw=0.8, zorder=6 if is_ego else 4))
    rx, ry, ryaw = recorded_rig[step]
    ghost = box_corners(rx + RIG_TO_BOX_M * np.cos(ryaw), ry + RIG_TO_BOX_M * np.sin(ryaw), ryaw, *box_sizes["EGO"])
    ax.add_patch(PolygonPatch(ghost, fill=False, edgecolor="tab:green", ls="--", lw=1.5, zorder=5))
    ax.plot(recorded_rig[:, 0], recorded_rig[:, 1], color="tab:green", lw=1.5, alpha=0.6, zorder=3)
    route_xy = route_in_local_frame(step_us[step])
    ax.scatter(route_xy[:, 0], route_xy[:, 1], color="tab:purple", s=8, zorder=3)
    if t_s >= 0:
        plan = plans[max(0, int(np.searchsorted(decision_s, t_s, side="right")) - 1)]
        plan_xy = np.array([[p.pose.vec.x, p.pose.vec.y] for p in plan])
        ax.plot(plan_xy[:, 0], plan_xy[:, 1], color="tab:orange", lw=2.5, zorder=7)
    ax.set_title(f"{name} ({t_s:+.1f} s), ego {deviation[step]:.1f} m from the recording", color=MOMENT_COLORS[name])


fig, axes = plt.subplots(2, 2, figsize=(15, 15))
for ax, (name, t_s) in zip(axes.flat, moments.items()):
    draw_moment(ax, name, t_s)
for ax in list(axes.flat)[len(moments):]:
    ax.axis("off")
handles = [
    PolygonPatch(np.zeros((4, 2)), facecolor="tab:green", edgecolor="k", label="ego"),
    PolygonPatch(np.zeros((4, 2)), fill=False, edgecolor="tab:green", ls="--", label="ego in the recording"),
    PolygonPatch(np.zeros((4, 2)), facecolor="0.75", edgecolor="k", label="other agents"),
    plt.Line2D([], [], color="tab:orange", lw=2.5, label="AutoE2E plan"),
    plt.Line2D([], [], color="tab:green", lw=1.5, label="recorded path"),
    plt.Line2D([], [], color="tab:purple", marker=".", ls="", label="route"),
]
fig.legend(handles=handles, loc="lower center", ncol=6)
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

### 6.5 Ego vs. recording

The recording shows what the human driver did from the same start. The panels below compare the ego with it at the
same time:

- speed
- heading
- **along-track lag:** how far behind the recording the ego is, measured along the recorded path
- **cross-track offset:** how far beside the recorded path the ego is, positive to the left

In [ ]:
from shapely.geometry import LineString, Point

recorded_line = LineString(
    [se3(recorded, int(t))[:2, 3] for t in np.arange(recorded_range.start, recorded_range.stop, 50_000)]
)


def along_and_across(xy):
    along, across = [], []
    for point in xy:
        s = recorded_line.project(Point(point))
        before = np.asarray(recorded_line.interpolate(max(s - 0.5, 0.0)).coords[0])
        after = np.asarray(recorded_line.interpolate(min(s + 0.5, recorded_line.length)).coords[0])
        foot = np.asarray(recorded_line.interpolate(s).coords[0])
        tangent, offset = after - before, point - foot
        along.append(s)
        across.append(np.sign(tangent[0] * offset[1] - tangent[1] * offset[0]) * np.hypot(*offset))
    return np.array(along), np.array(across)


along_ego, across_ego = along_and_across(ego_rig[:, :2])
along_recorded, _ = along_and_across(recorded_rig[:, :2])
time_s = step_us / 1e6
speed_ego = np.hypot(*np.gradient(ego_rig[:, :2], time_s, axis=0).T)
speed_recorded = np.hypot(*np.gradient(recorded_rig[:, :2], time_s, axis=0).T)
heading_error = np.degrees(np.angle(np.exp(1j * (ego_rig[:, 2] - recorded_rig[:, 2]))))

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex=True)
axes[0, 0].plot(t_steps, speed_recorded, color="0.4", label="recording")
axes[0, 0].plot(t_steps, speed_ego, color="tab:blue", label="ego")
axes[0, 0].set(title="Speed", ylabel="m/s")
axes[0, 1].plot(t_steps, heading_error, color="tab:blue")
axes[0, 1].set(title="Heading, ego minus recording", ylabel="degrees")
axes[1, 0].plot(t_steps, along_recorded - along_ego, color="tab:blue")
axes[1, 0].set(title="Along-track lag behind the recording", ylabel="m", xlabel="seconds after takeover")
axes[1, 1].plot(t_steps, across_ego, color="tab:blue")
axes[1, 1].set(title="Cross-track offset from the recorded path", ylabel="m (left +)", xlabel="seconds after takeover")
for ax in axes.flat:
    mark_moments(ax)
    ax.axvspan(t_steps[0], 0, color="0.9", zorder=0)
axes[0, 0].legend()
fig.legend(handles=[plt.Line2D([], [], color=c, ls="--", label=n) for n, c in MOMENT_COLORS.items() if n in moments]
           + [PolygonPatch(np.zeros((4, 2)), color="0.9", label="warm-up (ego replays the recording)")],
           loc="lower center", ncol=5)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

Next, each plan against the recorded future over the same 6.4 s, in the ego's rig frame at decision time (forward
is up). This is what the reward's tracking term measures. The FDE is computed at the last recorded waypoint.

In [ ]:
def plan_and_recorded_future(poses):
    plan = to_trajectory(poses)
    rig_from_local = np.linalg.inv(se3(plan, int(poses[0].timestamp_us)))
    plan_xy = np.array([(rig_from_local @ se3(plan, int(p.timestamp_us)))[:2, 3] for p in poses[1:]])
    future_xy = np.array(
        [(rig_from_local @ se3(recorded, int(p.timestamp_us)))[:2, 3] for p in poses[1:] if int(p.timestamp_us) in recorded_range]
    )
    return plan_xy, future_xy


chosen = np.unique(np.linspace(0, len(plans) - 1, 8).astype(int))
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, index in zip(axes.flat, chosen):
    plan_xy, future_xy = plan_and_recorded_future(plans[index])
    errors = np.hypot(*(plan_xy[: len(future_xy)] - future_xy).T)
    ax.plot(-future_xy[:, 1], future_xy[:, 0], color="tab:green", lw=2.5, label="recorded future")
    ax.plot(-plan_xy[:, 1], plan_xy[:, 0], color="tab:orange", lw=2, label="AutoE2E plan")
    ax.scatter([0], [0], color="k", marker="^", zorder=5)
    points = np.vstack([plan_xy, future_xy, [[0.0, 0.0]]])
    center = (points.max(axis=0) + points.min(axis=0)) / 2
    half = max(10.0, (points.max(axis=0) - points.min(axis=0)).max() / 2 + 2.0)
    ax.set_xlim(-center[1] - half, -center[1] + half)
    ax.set_ylim(center[0] - half, center[0] + half)
    ax.set(title=f"decision {index + 1} ({decision_s[index]:+.1f} s)\nADE {errors.mean():.1f} m, FDE {errors[-1]:.1f} m",
           xlabel="right (m)", ylabel="forward (m)", aspect="equal")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

### 6.6 Infractions and AlpaSim's metrics

AlpaSim scores every simulation step (`metrics.parquet`). Its summary table aggregates those scores only after
cutting the timeline:

1. it drops the warm-up
2. it stops at the first off-road or collision, or once the ego is 4 m from the recorded trajectory, whichever
   comes first

So "offroad 0" in the summary means no off-road step **before the cut**. The per-step data below covers the whole
rollout.

In [ ]:
events = [name for name in ("offroad", "collision_any", "wrong_lane", "open_loop_collision", "safety_monitor_triggered") if name in metrics]
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, gridspec_kw={"height_ratios": [1.3, 1, 1, 1]})
for row, name in enumerate(events):
    hits = metrics.loc[metrics[name] > 0, "t"]
    axes[0].scatter(hits, np.full(len(hits), row), marker="|", s=300, color="tab:red")
axes[0].set_yticks(range(len(events)), events)
axes[0].set_ylim(-0.7, len(events) - 0.3)
axes[0].set_title("Steps AlpaSim flagged (one tick per 0.1 s step); open_loop_collision means a plan would collide if driven as is")
axes[1].plot(metrics["t"], metrics["dist_to_gt_trajectory"], color="tab:blue", label="distance to the recorded trajectory")
axes[1].axhline(4.0, color="tab:purple", ls=":", label="AlpaSim's evaluation cut-off (4 m)")
axes[1].axhline(3.0, color="tab:orange", ls=":", label="phase-1 reward bound (3 m)")
axes[1].set(ylabel="m")
axes[1].legend(loc="upper left")
axes[2].plot(metrics["t"], metrics["min_distance_to_lane_boundary_m"], label="to the nearest lane boundary")
axes[2].plot(metrics["t"], metrics["min_distance_to_obstacle_m"], label="to the nearest obstacle")
axes[2].set(ylabel="m", title="Clearances")
axes[2].legend(loc="upper left")
# AlpaSim's gt_dist_traveled_m is the recording's total, so the running distances come from the poses.
axes[3].plot(t_steps, np.r_[0, np.cumsum(np.hypot(*np.diff(recorded_rig[:, :2], axis=0).T))], color="0.4", label="recording")
axes[3].plot(t_steps, np.r_[0, np.cumsum(np.hypot(*np.diff(ego_rig[:, :2], axis=0).T))], color="tab:blue", label="ego")
axes[3].axhline(metrics["gt_dist_traveled_m"].iloc[-1], color="0.4", ls=":", label="whole recording")
axes[3].set(ylabel="m", title="Distance travelled since the start of the rollout", xlabel="seconds after takeover")
axes[3].legend(loc="upper left")
for ax in axes:
    mark_moments(ax)
fig.tight_layout()
plt.show()

In [ ]:
driving = metrics[metrics["t"] >= 0]
cuts = [t for t in (first_time(driving["t"], (driving["offroad"] > 0) | (driving["collision_any"] > 0)),
                    first_time(driving["t"], driving["dist_to_gt_trajectory"] >= 4.0)) if t is not None]
evaluated = driving[driving["t"] <= min(cuts)] if cuts else driving


def summary(rows) -> dict:
    return {
        "steps": len(rows),
        "offroad (max)": rows["offroad"].max(),
        "collision_any (max)": rows["collision_any"].max(),
        "dist_to_gt_trajectory (max, m)": round(rows["dist_to_gt_trajectory"].max(), 2),
        "distance travelled (m)": round(rows["dist_traveled_m"].iloc[-1], 2),
        "progress (last)": round(rows["progress"].iloc[-1], 2),
    }


print(f"AlpaSim's evaluation stops at {min(cuts):+.1f} s" if cuts else "AlpaSim's evaluation covers the whole rollout")
pd.DataFrame({"whole rollout": summary(driving), "what AlpaSim's summary counts": summary(evaluated)})

AlpaSim's own summary chart, from `aggregate/metrics_results.png`:

In [ ]:
display(DisplayImage(filename=str(ROLLOUT_LOG_DIR / "aggregate" / "metrics_results.png")))

### 6.7 What the cameras saw

The log keeps every frame the driver received: one `driver_camera_image` message per camera per 0.1 s step. These are
the observations an RL update would train on. First, all six cameras at the key moments, in the model's slot order.
The frames are NuRec renders of NVIDIA's dataset, so the committed copy of this notebook keeps no outputs.

In [ ]:
frame_times = collections.defaultdict(list)
for camera, end_us in frames:
    frame_times[camera].append(end_us)
frame_times = {camera: np.array(sorted(times)) for camera, times in frame_times.items()}


def frame_at(camera: str, t_us: int, width: int = 480) -> np.ndarray:
    """The camera's newest frame that ended at or before t_us, decoded to RGB and resized to `width`."""
    times = frame_times[camera]
    end_us = times[max(0, int(np.searchsorted(times, t_us, side="right")) - 1)]
    image = cv2.imdecode(np.frombuffer(frames[(camera, int(end_us))], np.uint8), cv2.IMREAD_COLOR)[..., ::-1]
    return cv2.resize(image, (width, int(image.shape[0] * width / image.shape[1])), interpolation=cv2.INTER_AREA)


fig, axes = plt.subplots(len(moments), len(rig), figsize=(4 * len(rig), 2.5 * len(moments)))
for row, (name, t_s) in enumerate(moments.items()):
    for col, camera in enumerate(rig):
        ax = axes[row, col]
        ax.imshow(frame_at(camera.logical_id, T0 + int(t_s * 1e6)))
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"{camera.slot}\n{camera.logical_id}", fontsize=9)
        if col == 0:
            ax.set_ylabel(f"{name}\n{t_s:+.1f} s", fontsize=9, color=MOMENT_COLORS[name])
fig.tight_layout()
plt.show()

Next, the front camera every second from the takeover. NuRec renders degrade as the ego moves away from the recorded
path, which is why the reward stops trusting the scene beyond 3 m.

In [ ]:
times = np.arange(0.0, t_steps[-1] + 1e-6, 1.0)
columns = 7
fig, axes = plt.subplots(int(np.ceil(len(times) / columns)), columns, figsize=(3.2 * columns, 2.1 * np.ceil(len(times) / columns)))
for ax, t_s in zip(axes.flat, times):
    step = int(np.argmin(np.abs(t_steps - t_s)))
    ax.imshow(frame_at(rig[0].logical_id, T0 + int(t_s * 1e6), width=400))
    ax.set_title(f"{t_s:+.0f} s, {deviation[step]:.1f} m off", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()

Finally, the frames that make up one decision's camera input: 8 frames per camera, 500 ms apart, ending at the
decision time. This is the first decision taken with the ego more than 3 m from the recording, or the first decision
if there is none.

In [ ]:
t_decision = moments.get("ego > 3 m from the recording", 0.0)
index = max(0, int(np.searchsorted(decision_s, t_decision, side="right")) - 1)
history_us = [int(decision_us[index]) - k * 500_000 for k in range(CONTEXT_FRAMES - 1, -1, -1)]
fig, axes = plt.subplots(len(history_us), len(rig), figsize=(3 * len(rig), 1.75 * len(history_us)))
for row, t_us in enumerate(history_us):
    for col, camera in enumerate(rig):
        ax = axes[row, col]
        ax.imshow(frame_at(camera.logical_id, t_us, width=320))
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(camera.slot, fontsize=9)
        if col == 0:
            ax.set_ylabel(f"t0 {(t_us - decision_us[index]) / 1e6:+.1f} s", fontsize=9)
fig.suptitle(f"Camera input of decision {index + 1} ({decision_s[index]:+.1f} s after takeover)")
fig.tight_layout()
plt.show()

## 7. The phase-1 reward

`GroundTruthDeviationReward` (`alpasim_autoe2e/rewards.py`) scores each decision. It is the reward agreed in #123 and
#177.

- **Tracking:** `-(1.0·ADE + 0.5·FDE)` between the plan and the recorded drive at the same timestamps, in the rig
  frame at decision time.
- **Renderer validity bound:** once the ego itself is more than 3 m from the recorded drive, a −10 penalty is added
  and the decision is flagged `truncated`. Beyond that distance the Gaussian-splat reconstruction is unreliable.

`score_rollout.score()` wraps the async log reader in `asyncio.run()`. That cannot run inside the notebook's own
event loop, so it runs in a worker thread here.

In [ ]:
from alpasim_autoe2e.rewards import GroundTruthDeviationReward
from score_rollout import score

with ThreadPoolExecutor(max_workers=1) as pool:
    rows = pool.submit(score, str(rollout_asl), GroundTruthDeviationReward()).result()
scores = pd.DataFrame(rows)
scores["time (s)"] = seconds(scores["t0_us"])
print(f"{len(scores)} decisions, {int(scores['truncated'].sum())} truncated, mean reward {scores['reward'].mean():.2f}")
scores[["ade_m", "fde_m", "ego_deviation_m", "tracking", "boundary", "reward"]].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
axes[0].plot(scores["time (s)"], scores["ade_m"], label="ADE")
axes[0].plot(scores["time (s)"], scores["fde_m"], label="FDE")
axes[0].set(title="Plan vs. recorded drive", xlabel="seconds after takeover", ylabel="m")
axes[0].legend()
axes[1].fill_between(scores["time (s)"], scores["tracking"], color="tab:blue", alpha=0.5, label="tracking term")
axes[1].bar(scores["time (s)"], scores["boundary"], width=0.1, color="tab:red", alpha=0.7, label="3 m bound penalty")
axes[1].plot(scores["time (s)"], scores["reward"], color="k", lw=1, label="reward")
axes[1].set(title="Phase-1 reward per decision, by term", xlabel="seconds after takeover")
axes[1].legend(loc="lower left")
axes[2].plot(scores["time (s)"], scores["reward"].cumsum(), color="tab:green")
truncated = scores[scores["truncated"]]
if len(truncated):
    axes[2].axvline(truncated["time (s)"].iloc[0], color="tab:orange", ls="--", label="first truncated decision")
    axes[2].legend()
axes[2].set(title="Cumulative reward (undiscounted)", xlabel="seconds after takeover")
fig.tight_layout()
plt.show()

## 8. From a rollout to fine-tuning data

An RL update learns from transitions. Here is the per-decision table this rollout yields. Each row has:

- when the decision was made, and where the ego was
- the plan the policy chose, as 64 waypoints in the rig frame
- its reward
- whether the episode should end there (the first truncation)

The observations are in `rollout.asl` too: the camera frames in part 6.7. A training loop can therefore pair each
decision with its inputs without re-rendering. At about 130 MB per 20 s rollout, though, storing them dominates the
cost of logging.

In [ ]:
def plan_in_rig_frame(poses) -> np.ndarray:
    return plan_and_recorded_future(poses)[0]


transitions = pd.DataFrame(
    {
        "decision": scores["decision"],
        "t0_us": scores["t0_us"],
        "ego_x": [plan[0].pose.vec.x for plan in plans],
        "ego_y": [plan[0].pose.vec.y for plan in plans],
        "ego_yaw": [yaw_of(plan[0].pose.quat) for plan in plans],
        "plan_xy_rig": [plan_in_rig_frame(plan) for plan in plans],
        "reward": scores["reward"],
        "truncated": scores["truncated"],
    }
)
first_truncation = transitions.index[transitions["truncated"]].min() if transitions["truncated"].any() else None
transitions["done"] = transitions.index >= first_truncation if first_truncation is not None else False
transitions["done"] |= transitions.index == transitions.index[-1]
print(f"episode length: {int((~transitions['done']).sum()) + 1} of {len(transitions)} decisions")
transitions.head()

### Where the fine-tuning update plugs in (planned)

Everything above runs today. The loop that would turn tables like this into a better policy does not exist yet. It
is the future work in `Design/reinforcement_learning.md` §6–7, and would need:

1. **An RL environment around AlpaSim.** It ends an episode at the first `truncated` decision instead of driving to
   the end of the scene, and turns the logged frames, plans and rewards into training samples.
2. **Rollout collection over many scenes.** This is either the full pixel simulator ("Tier-P", this notebook) or
   the faster latent world-model rollouts ("Tier-L", `SimAdapter`).
3. **A policy update.** It keeps imitation learning as a regularisation term, following RAD. The phase-2 reward
   terms (safety, progress, comfort, faithfulness-gated reasoning) come from a `RewardRegistry`.
4. **Re-evaluation of the new checkpoint** in the same closed loop.

The design document does not choose an update algorithm. This notebook deliberately stops at the data such an update
would consume.